In [36]:
import polars as pl
import pandas as pd
import numpy as np
import os

DATA_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\dataset\sales_pers.item_chunk_0.parquet"

df = pl.read_parquet(DATA_PATH)
print(f"Số dòng: {df.height:,}, Số cột: {df.width}")


Số dòng: 27,332, Số cột: 34


In [37]:
print("\nCác cột ban đầu:")
print(df.columns)


Các cột ban đầu:
['p_id', 'item_id', 'price', 'category_l1_id', 'category_l1', 'category_l2_id', 'category_l2', 'category_l3_id', 'category_l3', 'category_id', 'category', 'description', 'brand', 'manufacturer', 'creation_timestamp', 'is_deleted', 'created_date', 'updated_date', 'sync_status_id', 'last_sync_date', 'sync_error_message', 'image_url', 'gender_target', 'age_group', 'item_type', 'gp', 'weight', 'color', 'size', 'origin', 'volume', 'material', 'sale_status', 'description_new']


# Task 1: Loại bỏ các cột mà nhóm nghĩ là không cần thiết.

In [38]:
# --- Danh sách cột cần loại bỏ (theo phân tích EDA & tương quan) ---
cols_to_drop = [
    # --- Metadata hệ thống (không dùng cho mô hình) ---
    "p_id",
    "is_deleted",
    "sync_status_id",
    "sync_error_message",
    "image_url",
    "last_sync_date",
    "creation_timestamp",
    "updated_date",
    "created_date",
    "sale_status",

    # --- Cột numeric hầu như vô nghĩa / chỉ có 1 giá trị ---
    "gp",
    "weight",
    "volume",

    # --- Các cột chất lượng kém / missing cực cao / không dùng ---
    "color",
    "size",
    "material",
    "origin",
    "manufacturer",

    # --- ID phân loại (trùng với tên category) ---
    "category_l1_id",
    "category_l2_id",
    "category_l3_id",
    "category_id"
]



# --- Loại bỏ các cột không cần thiết ---
df_cleaned = df.drop(cols_to_drop)

print(f"\nĐã loại bỏ {len(cols_to_drop)} cột không cần thiết.")
print(f"Số cột còn lại: {df_cleaned.width}")
print("\nDanh sách cột sau khi loại bỏ:")
print(df_cleaned.columns)



Đã loại bỏ 22 cột không cần thiết.
Số cột còn lại: 12

Danh sách cột sau khi loại bỏ:
['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'description', 'brand', 'gender_target', 'age_group', 'item_type', 'description_new']


# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lý outlier

In [39]:
# df_cleaned: dataframe sau khi đã drop các cột không cần thiết
print("Số lượng ban đầu:", df_cleaned.height)

# --- LOẠI SẢN PHẨM CÓ PRICE ≤ 1 (Outlier) ---
df_no_outlier = df_cleaned.filter(pl.col("price") > 1)

print("Số lượng sau khi loại outlier (price ≤ 1):", df_no_outlier.height)
print("Số lượng bị loại:", df_cleaned.height - df_no_outlier.height)

# Kiểm tra lại xem còn outlier hay không
print("\nKiểm tra min price sau khi xử lý:")
print(df_no_outlier.select(pl.col("price").min()))

Số lượng ban đầu: 27332
Số lượng sau khi loại outlier (price ≤ 1): 27323
Số lượng bị loại: 9

Kiểm tra min price sau khi xử lý:
shape: (1, 1)
┌───────────────┐
│ price         │
│ ---           │
│ decimal[38,4] │
╞═══════════════╡
│ 1000.0000     │
└───────────────┘


In [40]:
# Chuẩn hóa giá trị Unisex → Không xác định
df_no_outlier = df_no_outlier.with_columns(
    pl.col("gender_target").replace("Unisex", "Không xác định")
)

print("Đã chuyển toàn bộ Unisex thành 'Không xác định'.")
print(df_no_outlier["gender_target"].value_counts())


Đã chuyển toàn bộ Unisex thành 'Không xác định'.
shape: (4, 2)
┌────────────────┬───────┐
│ gender_target  ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ Bé Trai        ┆ 3318  │
│ Không xác định ┆ 18035 │
│ Bé Gái         ┆ 4108  │
│ Sơ sinh        ┆ 1862  │
└────────────────┴───────┘


In [41]:
df_no_outlier.head(5)

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …"
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …"
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""


## Xử lý null

In [15]:
pl.Config.set_tbl_rows(50)             # số dòng tối đa hiển thị


# df_final = df_no_outlier
df_final = df_no_outlier

n = df_final.height

results = []

for col in df_final.columns:
    s = df_final[col]

    # NULL
    null_count = s.null_count()

    # UNKNOWN ("Không xác định")
    if s.dtype == pl.String:
        # Đếm trực tiếp bằng Python list, không lỗi
        unknown_count = s.to_list().count("Không xác định")
    else:
        unknown_count = 0

    results.append({
        "column": col,
        "dtype": s.dtype,
        "null_count": null_count,
        "null_ratio(%)": round(null_count / n * 100, 2),
        "unknown_count": unknown_count,
        "unknown_ratio(%)": round(unknown_count / n * 100, 2)
    })

df_missing_report = pl.DataFrame(results)

print(df_missing_report)


shape: (13, 6)
┌─────────────────┬─────────────────┬────────────┬───────────────┬───────────────┬─────────────────┐
│ column          ┆ dtype           ┆ null_count ┆ null_ratio(%) ┆ unknown_count ┆ unknown_ratio(% │
│ ---             ┆ ---             ┆ ---        ┆ ---           ┆ ---           ┆ )               │
│ str             ┆ object          ┆ i64        ┆ f64           ┆ i64           ┆ ---             │
│                 ┆                 ┆            ┆               ┆               ┆ f64             │
╞═════════════════╪═════════════════╪════════════╪═══════════════╪═══════════════╪═════════════════╡
│ item_id         ┆ String          ┆ 0          ┆ 0.0           ┆ 0             ┆ 0.0             │
│ price           ┆ Decimal(precisi ┆ 0          ┆ 0.0           ┆ 0             ┆ 0.0             │
│                 ┆ on=38, scale=4) ┆            ┆               ┆               ┆                 │
│ category_l1     ┆ String          ┆ 0          ┆ 0.0           ┆ 0        

### Xử lý gender_target

In [42]:

# Lọc các item thời trang nhưng gender_target = "Không xác định"
df_fashion_unknown = (
    df_no_outlier
        .filter(
            (pl.col("category_l1") == "Thời trang") &
            (pl.col("gender_target") == "Không xác định")
        )
)

print("Tổng số item thời trang có gender_target = 'Không xác định':", df_fashion_unknown.height)

# Lấy 20 dòng ngẫu nhiên
df_sample = df_fashion_unknown.sample(n=20, with_replacement=False)

pd.set_option('display.max_columns', None)
df_sample.head(10)

Tổng số item thời trang có gender_target = 'Không xác định': 7264


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""4564000000368""",189000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bộ Modal""","""Bộ Modal lẻ""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0024171040001""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Quần sơ sinh""","""Set 2 Quần sơ sinh dài Concung…","""Con Cưng""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""3532000000263""",119000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Quần""","""Quần sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0878104790001""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Đầm""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3324000000405""",219000.0000,"""Thời trang""","""Thời trang bé gái""","""Đầm bé gái""","""Đầm bé gái Animo""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""4564000000370""",189000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bộ Modal""","""Bộ Modal lẻ""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""6997000000244""",349000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bộ Modal""","""Bộ Modal set 2""","""﻿﻿Set 2 Bộ kháng khuẩn Modal n…","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0858019490004""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần bé trai""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3953000000768""",129000.0000,"""Thời trang""","""Thời trang bé trai""","""Bộ bé trai""","""Bộ bé trai Animo Easy""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""


In [43]:
# Lọc sản phẩm thời trang có category chứa "đầm" hoặc "váy" và gender_target = "Không xác định"
df_dam_vay_unknown = (
    df_no_outlier
    .filter(
        (pl.col("category_l1") == "Thời trang")
        &
        (pl.col("gender_target") == "Không xác định")
        &
        (
            pl.col("category").str.contains("đầm", literal=False)
            | pl.col("category").str.contains("váy", literal=False)
        )
    )
)

# In thống kê số dòng
print("Tổng sản phẩm thỏa điều kiện:", df_dam_vay_unknown.height)

# Lấy mẫu 20 dòng
df_sample = df_dam_vay_unknown.sample(n=20, seed=42)

# Hiển thị đầy đủ cột
pd.set_option('display.max_columns', None)
df_sample.head(10)


Tổng sản phẩm thỏa điều kiện: 192


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""6045000000002""",349000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""1092093850001""",189000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Áo bầu""","""Không xác định"""
"""1083022850002""",149000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Quần bầu""","""Không xác định"""
"""0886026770002""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0887026770004""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0891019790002""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0012190160065""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""Laluna""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""1080033860001""",199000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""1080004860001""",189000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Quần, áo, đầm bầu""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Mẹ""","""Áo bầu""","""Không xác định"""


In [46]:
# Sao chép gender_target sang cột mới
df_filled = df_no_outlier.with_columns(
    pl.col("gender_target").alias("gender_target_final")
)

# =========================
# 1) FILL SƠ SINH
# =========================

mask_sosinh = (
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("gender_target_final") == "Không xác định")
    & (
        pl.col("category_l2").str.contains("sơ sinh", literal=False)
        | pl.col("category").str.contains("sơ sinh", literal=False)
        | pl.col("category").str.contains("0-3m", literal=False)
        | pl.col("category").str.contains("3-6m", literal=False)
        | pl.col("category").str.contains("0-12m", literal=False)
        | pl.col("category").str.contains("\\bnb\\b", literal=False)  # NB
        | pl.col("category").str.contains("newborn", literal=False)
    )
)

df_filled = df_filled.with_columns(
    pl.when(mask_sosinh)
      .then("Sơ sinh")
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)

# =========================
# 2) FILL BÉ GÁI
# =========================

mask_fashion_unknown = (
    (pl.col("category_l1") == "Thời trang")
    & (pl.col("gender_target_final") == "Không xác định")
)

mask_not_maternity = (
    (~pl.col("category").str.contains("bầu", literal=False))
    & (~pl.col("category_l2").str.contains("bầu", literal=False))
    & (~pl.col("category_l3").str.contains("bầu", literal=False))
)

mask_not_ambiguous = (
    ~pl.col("category_l3").str.contains("Thời trang bé trai, bé gái cũ", literal=False)
)

mask_girl_keyword = (
    pl.col("category").str.contains("bé gái|đầm|váy|chân váy", literal=False)
    | pl.col("category_l2").str.contains("bé gái|đầm bé gái|bộ bé gái", literal=False)
    | pl.col("category_l3").str.contains("bé gái|đầm|váy", literal=False)
)

mask_fill_girl = mask_fashion_unknown & mask_not_maternity & mask_not_ambiguous & mask_girl_keyword

df_filled = df_filled.with_columns(
    pl.when(mask_fill_girl)
      .then("Bé Gái")
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)

# =========================
# 3) FILL BÉ TRAI
# =========================

mask_boy_keyword = (
    pl.col("category").str.contains("bé trai", literal=False)
    | pl.col("category_l2").str.contains("bé trai|bộ bé trai", literal=False)
    | pl.col("category_l3").str.contains("bé trai", literal=False)
)

mask_fill_boy = mask_fashion_unknown & mask_not_ambiguous & mask_boy_keyword

df_filled = df_filled.with_columns(
    pl.when(mask_fill_boy)
      .then("Bé Trai")
      .otherwise(pl.col("gender_target_final"))
      .alias("gender_target_final")
)


ColumnNotFoundError: unable to find column "Sơ sinh"; valid columns: ["item_id", "price", "category_l1", "category_l2", "category_l3", "category", "description", "brand", "gender_target", "age_group", "item_type", "description_new", "gender_target_final"]

In [44]:

# Lọc các sản phẩm thuộc Phụ kiện và gender_target = "Không xác định"
df_phukien_unknown = (
    df_no_outlier
    .filter(
        (pl.col("category_l1") == "Phụ kiện") &
        (pl.col("gender_target") == "Không xác định")
    )
)

# Kiểm tra số lượng
print("Tổng số sản phẩm Phụ kiện có gender_target = 'Không xác định':",
      df_phukien_unknown.height)

# Lấy ngẫu nhiên 10 dòng
df_sample = df_phukien_unknown.sample(n=10, seed=42)

# In ra toàn bộ cột để bạn xem xét thật chi tiết
pd.set_option('display.max_columns', None)
df_sample.head(10)


Tổng số sản phẩm Phụ kiện có gender_target = 'Không xác định': 1666


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""6053000000007""",49000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Giày dép tồn""","""Giày bún tập đi""","""﻿Giày bún tập đi Animo A2206_J…","""Animo""","""Không xác định""","""1Y-3Y""","""Giày""","""Chi tiết sản phẩm …"
"""6055000000002""",49000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Giày dép tồn""","""Giày bún tập đi""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Giày""","""Chi tiết sản phẩm …"
"""4804000000016""",249000.0000,"""Phụ kiện""","""Phụ kiện khác""","""Túi xách, Ba lô""","""Ba lô""","""Không xác định""","""Mesuca""","""Không xác định""","""Từ 2Y""","""Balo""","""Chi tiết sản phẩm …"
"""4794000000015""",139000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Phụ kiện tồn""","""Phụ kiện tồn SPC""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""4812000000011""",99000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Giày dép tồn""","""Dép sục người lớn""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""6055000000036""",119000.0000,"""Phụ kiện""","""Giày tập đi""","""Giày sơ sinh 119k""","""Giày sơ sinh 119k S11""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …"
"""0014521040104""",9000.0000,"""Phụ kiện""","""Cơ cấu hàng cũ""","""Phụ kiện tồn""","""Phụ kiện khác tồn""","""Không xác định""","""CF (ConCung Fashion)""","""Không xác định""","""Không xác định""","""Kẹp tóc""","""Chi tiết sản phẩm …"
"""4003000000014""",229000.0000,"""Phụ kiện""","""Giày tập đi""","""Giày chút chít 179k""","""Giày chút chít 229k S15""","""﻿﻿Giày tập đi chút chít Animo …","""Animo""","""Không xác định""","""0-24M""","""Giày tập đi""","""Chi tiết sản phẩmTên sản phẩm:…"
"""4027000000049""",139000.0000,"""Phụ kiện""","""Nón""","""Nón sơ sinh""","""0-12M Nón khăn voan""","""﻿﻿Nón vành tròn khăn voan bé t…","""Animo""","""Không xác định""","""0-12M""","""Nón""","""Chi tiết sản phẩm …"


### Xử lý age_group

In [45]:
# Lọc các sản phẩm Thời trang nhưng age_group = "Không xác định"
df_fashion_age_unknown = (
    df_no_outlier
    .filter(
        (pl.col("category_l1") == "Thời trang") &
        (pl.col("age_group") == "Không xác định")
    )
)

# Kiểm tra số lượng
total = df_fashion_age_unknown.height
print("Tổng số sản phẩm Thời trang có age_group = 'Không xác định':", total)

# -----------------------------
# Thống kê description & description_new
# -----------------------------

count_desc_unknown = df_fashion_age_unknown.filter(pl.col("description") == "Không xác định").height
count_desc_new_unknown = df_fashion_age_unknown.filter(pl.col("description_new") == "Không xác định").height

print("\nThống kê tình trạng mô tả trong nhóm này:")
print(f"- description = 'Không xác định': {count_desc_unknown} / {total} ({count_desc_unknown/total*100:.2f}%)")
print(f"- description_new = 'Không xác định': {count_desc_new_unknown} / {total} ({count_desc_new_unknown/total*100:.2f}%)")

# -----------------------------
# Lấy ngẫu nhiên 20 dòng để quan sát
# -----------------------------
df_sample_age = df_fashion_age_unknown.sample(n=20, seed=42)

pd.set_option('display.max_columns', None)
df_sample_age.head(20)


Tổng số sản phẩm Thời trang có age_group = 'Không xác định': 6058

Thống kê tình trạng mô tả trong nhóm này:
- description = 'Không xác định': 5435 / 6058 (89.72%)
- description_new = 'Không xác định': 2593 / 6058 (42.80%)


item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str
"""3523000000021""",99000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Áo""","""Áo sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""3320016830007""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé trai""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""0930002360001""",19000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Quần, áo & phụ kiện sơ sinh cũ""","""Tã vải""","""Không xác định""","""CF (ConCung Fashion)""","""Sơ sinh""","""Không xác định""","""Tã vải""","""Không xác định"""
"""3533000000241""",119000.0000,"""Thời trang""","""Quần áo & Phụ kiện sơ sinh""","""Quần""","""Quần sơ sinh Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null
"""6034000000360""",129000.0000,"""Thời trang""","""Thời trang bé gái""","""Bộ bé gái""","""Bộ bé gái Animo Easy""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""3951000000444""",139000.0000,"""Thời trang""","""Thời trang bé trai""","""Bộ bé trai""","""Bộ bé trai Animo Easy""","""﻿﻿Bộ bé trai ngắn Animo Easy H…","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0891104050001""",49000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Quần, chân váy bé gái""","""Không xác định""","""CF (ConCung Fashion)""","""Bé Gái""","""Không xác định""","""Quần""","""Không xác định"""
"""6998000000086""",379000.0000,"""Thời trang""","""Modal kháng khuẩn""","""Bộ Modal""","""Bộ chống muỗi set 2""","""Không xác định""","""Animo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""3414016830002""",89000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé trai""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",null


# Task 3: Phân tích tương đồng và xác định xem các thuộc tính tương tự nhau. Từ đó loại bỏ đặc trưng thừa.

# Task 4: Chuẩn hóa dữ liệu (nếu có), biến đổi dữ liệu

# Task 5: Nhóm hãy suy nghĩ xem, với bài toán dự đoán mua hàng, ta có thể tạo mới những đặc trưng nào. Sau đó tiến hành rút trích thêm các đặc trưng. Task này rất quan trọng vì ảnh hưởng hiệu quả của hệ thống.